In [ ]:
# Import libraries
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

pd.set_option('display.max_columns', None)

/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [ ]:
# set data dir
DATA_DIR = Path('')
os.listdir(DATA_DIR)

['Train.csv', 'Test.csv', 'SampleSubmission.csv']

In [ ]:
# Load files
train = pd.read_csv(DATA_DIR / 'Train.csv')
test = pd.read_csv(DATA_DIR / 'Test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')

# Preview train file
train.head()

,ID,time,Green,Blue,RED,NIR,SWIR1,SWIR2,Red_Edge,Aerosols,Red_Edge_2,Red_Edge_3,Red_Edge_4,Water_vapor,Cirrus,NDVI,NDMI,NDWI,CI,Target
0,1D_0000,2021-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1D_0000,2021-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1D_0000,2021-01-14,0.0845,0.1030,0.0583,0.1977,0.1152,0.0464,0.0742,0.1326,0.1682,0.2243,0.2500,0.0347,0.0007,0.544531,0.263663,-0.401134,0.120000,0
3,1D_0000,2021-01-19,0.1095,0.1357,0.0858,0.1596,0.0782,0.0330,0.0873,0.1620,0.1396,0.1736,0.1820,0.0260,0.0011,0.300733,0.342304,-0.186176,0.008666,0
4,1D_0000,2021-01-24,0.1128,0.1300,0.0919,0.2141,0.1240,0.0608,0.1035,0.1546,0.1845,0.2302,0.2606,0.0703,0.0041,0.399346,0.266489,-0.309881,0.059365,0


In [ ]:
# Preview test file
test.head()

,ID,time,Green,Blue,RED,NIR,SWIR1,SWIR2,Red_Edge,Aerosols,Red_Edge_2,Red_Edge_3,Red_Edge_4,Water_vapor,Cirrus,NDVI,NDMI,NDWI,CI
0,1D_0005,2021-01-04,0.1199,0.1391,0.0978,0.2329,0.1782,0.0989,0.1146,0.2127,0.2074,0.2675,0.3081,0.0503,0.0011,0.408527,0.133058,-0.320295,0.079096
1,1D_0005,2021-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1D_0005,2021-01-14,0.0818,0.1010,0.0563,0.1848,0.1139,0.0459,0.0756,0.1324,0.1687,0.2246,0.2513,0.0372,0.0007,0.532974,0.237362,-0.386347,0.146323
3,1D_0005,2021-01-19,0.1270,0.1581,0.1031,0.1899,0.1186,0.0583,0.1124,0.1763,0.1639,0.1987,0.2156,0.0298,0.0012,0.296246,0.231118,-0.198485,0.043155
4,1D_0005,2021-01-24,0.1122,0.1311,0.0916,0.2125,0.1259,0.0608,0.1060,0.1553,0.1840,0.2307,0.2611,0.0709,0.0046,0.397567,0.255910,-0.308901,0.072874


In [ ]:
# Preview sample submission
sample_submission.head()

,ID,Target
0,1D_0005,0
1,1D_000A,0
2,1D_000D,0
3,1D_000E,0
4,1D_0015,0


In [ ]:
# Check shapes of files
train.shape, test.shape, sample_submission.shape

((11544707, 20), (4946879, 19), (16960, 2))

In [ ]:
# Check for duplicates and missing values - train
train.duplicated().any(), train.isnull().sum().any() # Handle missing values by filling them with meadian or any other techniques

(False, True)

In [ ]:
# Check for duplicates and missing values - test
test.duplicated().any(), test.isnull().sum().any() # Handle missing values by filling them with meadian or any other techniques

(False, True)

In [ ]:
# Check unique ids per dataset
train.ID.nunique(), test.ID.nunique()

(39571, 16960)

In [ ]:
# A single ID represents a unique pixel over different time periods
# create a targets_df

train_targets_df = train.groupby('ID')['Target'].first().reset_index()
train_targets_df.head()

,ID,Target
0,1D_0000,0
1,1D_0001,0
2,1D_0002,0
3,1D_0003,0
4,1D_0004,0


In [ ]:
# Check data distribution
train_targets_df.Target.value_counts(normalize = True)

,proportion
Target,
0,0.433398
1,0.335119
2,0.231483


In [ ]:
train.head()

,ID,time,Green,Blue,RED,NIR,SWIR1,SWIR2,Red_Edge,Aerosols,Red_Edge_2,Red_Edge_3,Red_Edge_4,Water_vapor,Cirrus,NDVI,NDMI,NDWI,CI,Target
0,1D_0000,2021-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1D_0000,2021-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1D_0000,2021-01-14,0.0845,0.1030,0.0583,0.1977,0.1152,0.0464,0.0742,0.1326,0.1682,0.2243,0.2500,0.0347,0.0007,0.544531,0.263663,-0.401134,0.120000,0
3,1D_0000,2021-01-19,0.1095,0.1357,0.0858,0.1596,0.0782,0.0330,0.0873,0.1620,0.1396,0.1736,0.1820,0.0260,0.0011,0.300733,0.342304,-0.186176,0.008666,0
4,1D_0000,2021-01-24,0.1128,0.1300,0.0919,0.2141,0.1240,0.0608,0.1035,0.1546,0.1845,0.2302,0.2606,0.0703,0.0041,0.399346,0.266489,-0.309881,0.059365,0


In [ ]:
# Aggrgate pixels over the different time periods to get a wholesome view
# Get columns to aggregate
agg_cols = train.columns.tolist()[2:-1]
agg_cols

['Green',
 'Blue',
 'RED',
 'NIR',
 'SWIR1',
 'SWIR2',
 'Red_Edge',
 'Aerosols',
 'Red_Edge_2',
 'Red_Edge_3',
 'Red_Edge_4',
 'Water_vapor',
 'Cirrus',
 'NDVI',
 'NDMI',
 'NDWI',
 'CI']

In [ ]:
# Define comprehensive aggregation functions
agg_functions = {
    col: [
        'mean',            # Arithmetic mean
        'std',             # Standard deviation
        'min',             # Minimum value
        'max',             # Maximum value
        'median',          # median
    ] for col in agg_cols
}

# Perform the aggregation
aggregated_train = train.groupby('ID').agg(agg_functions)
aggregated_test = test.groupby('ID').agg(agg_functions)

# Rename columns
aggregated_train.columns = [f"{col[0]}_{col[1]}" for col in aggregated_train.columns]
aggregated_test.columns = [f"{col[0]}_{col[1]}" for col in aggregated_test.columns]

# Reset index
aggregated_train = aggregated_train.reset_index()
aggregated_test = aggregated_test.reset_index()

# Preview aggregated train
aggregated_train.head()

,ID,Green_mean,Green_std,Green_min,Green_max,Green_median,Blue_mean,Blue_std,Blue_min,Blue_max,Blue_median,RED_mean,RED_std,RED_min,RED_max,RED_median,NIR_mean,NIR_std,NIR_min,NIR_max,NIR_median,SWIR1_mean,SWIR1_std,SWIR1_min,SWIR1_max,SWIR1_median,SWIR2_mean,SWIR2_std,SWIR2_min,SWIR2_max,SWIR2_median,Red_Edge_mean,Red_Edge_std,Red_Edge_min,Red_Edge_max,Red_Edge_median,Aerosols_mean,Aerosols_std,Aerosols_min,Aerosols_max,Aerosols_median,Red_Edge_2_mean,Red_Edge_2_std,Red_Edge_2_min,Red_Edge_2_max,Red_Edge_2_median,Red_Edge_3_mean,Red_Edge_3_std,Red_Edge_3_min,Red_Edge_3_max,Red_Edge_3_median,Red_Edge_4_mean,Red_Edge_4_std,Red_Edge_4_min,Red_Edge_4_max,Red_Edge_4_median,Water_vapor_mean,Water_vapor_std,Water_vapor_min,Water_vapor_max,Water_vapor_median,Cirrus_mean,Cirrus_std,Cirrus_min,Cirrus_max,Cirrus_median,NDVI_mean,NDVI_std,NDVI_min,NDVI_max,NDVI_median,NDMI_mean,NDMI_std,NDMI_min,NDMI_max,NDMI_median,NDWI_mean,NDWI_std,NDWI_min,NDWI_max,NDWI_median,CI_mean,CI_std,CI_min,CI_max,CI_median
0,1D_0000,0.265979,0.216543,0.0640,0.8526,0.13740,0.286144,0.217593,0.0859,0.8619,0.15790,0.259690,0.241008,0.0404,0.9137,0.11680,0.367115,0.188226,0.0832,0.9125,0.29625,0.243527,0.149443,0.0322,0.6009,0.16805,0.151610,0.113906,0.0142,0.4322,0.09440,0.263200,0.224953,0.0432,0.8770,0.13665,0.307138,0.205845,0.1181,0.8725,0.19780,0.340618,0.202055,0.0756,0.9129,0.24905,0.399149,0.202998,0.0975,0.9909,0.33135,0.426868,0.202845,0.0933,1.0262,0.36650,0.087540,0.077888,0.0149,0.4551,0.05885,0.009998,0.024321,0.0005,0.2330,0.00190,0.302709,0.232620,-0.031731,0.712057,0.318187,0.231466,0.105327,-0.054230,0.614286,0.254172,-0.243403,0.168868,-0.562095,0.007948,-0.245152,0.051841,0.072480,-0.056774,0.284581,0.037170
1,1D_0001,0.270399,0.208321,0.0638,0.8756,0.16675,0.289762,0.209404,0.0845,0.8789,0.18900,0.264174,0.232313,0.0396,0.9464,0.14545,0.379766,0.173545,0.0873,0.8901,0.32950,0.251989,0.142228,0.0332,0.6149,0.19385,0.156691,0.110389,0.0141,0.4389,0.11735,0.267897,0.215786,0.0441,0.8825,0.16175,0.309484,0.198924,0.1168,0.8528,0.20855,0.348151,0.190508,0.0737,0.9015,0.27855,0.408359,0.189984,0.0966,0.9757,0.36140,0.437205,0.190445,0.0992,1.0090,0.39655,0.087958,0.076198,0.0143,0.4441,0.06040,0.010103,0.025336,0.0004,0.2357,0.00205,0.301717,0.238602,-0.050286,0.761315,0.304885,0.229815,0.110257,0.010700,0.611240,0.251263,-0.248447,0.179584,-0.617116,0.026412,-0.227254,0.048843,0.071093,-0.045155,0.236865,0.027336
2,1D_0002,0.266084,0.216040,0.0627,0.8610,0.14775,0.286621,0.216439,0.0868,0.8711,0.16585,0.260865,0.241258,0.0398,0.9292,0.12585,0.366290,0.186754,0.0824,0.9086,0.30035,0.244240,0.148668,0.0324,0.5857,0.17620,0.153181,0.113040,0.0146,0.4329,0.10300,0.264412,0.225976,0.0429,0.8909,0.13765,0.307138,0.205845,0.1181,0.8725,0.19780,0.340472,0.203677,0.0718,0.9280,0.25095,0.398143,0.203926,0.0884,1.0060,0.32395,0.426066,0.204229,0.0902,1.0408,0.36305,0.087540,0.077888,0.0149,0.4551,0.05885,0.009998,0.024321,0.0005,0.2330,0.00190,0.299347,0.231626,-0.026485,0.721873,0.309474,0.229222,0.106075,-0.037047,0.612524,0.253078,-0.242578,0.168902,-0.582704,-0.001007,-0.226221,0.049452,0.071069,-0.080468,0.300748,0.034821
3,1D_0003,0.266266,0.216637,0.0642,0.8625,0.14630,0.286324,0.216656,0.0860,0.8722,0.16470,0.261431,0.241553,0.0406,0.9350,0.11895,0.361451,0.190320,0.0751,0.9100,0.29475,0.244240,0.148668,0.0324,0.5857,0.17620,0.153181,0.113040,0.0146,0.4329,0.10300,0.264412,0.225976,0.0429,0.8909,0.13765,0.307138,0.205845,0.1181,0.8725,0.19780,0.340472,0.203677,0.0718,0.9280,0.25095,0.398143,0.203926,0.0884,1.0060,0.32395,0.426066,0.204229,0.0902,1.0408,0.36305,0.087540,0.077888,0.0149,0.4551,0.05885,0.009998,0.024321,0.0005,0.2330,0.00190,0.289909,0.225776,-0.047141,0.716433,0.292897,0.219154,0.099812,-0.041223,0.612870,0.237944,-0.233115,0.162751,-0.549183,-0.002660,-0.211015,0.048114,0.066958,-0.040092,0.247040,0.027005
4,1D_0004,0.267871,0.215886,0.0632,0.8804,0.16230,0.287685,0.216091,0.0862,0.9002,0.17935,0.262368,0.241414,0.0410,0.9582,0.1448

In [ ]:
# Preview aggregated test
aggregated_test.head()

,ID,Green_mean,Green_std,Green_min,Green_max,Green_median,Blue_mean,Blue_std,Blue_min,Blue_max,Blue_median,RED_mean,RED_std,RED_min,RED_max,RED_median,NIR_mean,NIR_std,NIR_min,NIR_max,NIR_median,SWIR1_mean,SWIR1_std,SWIR1_min,SWIR1_max,SWIR1_median,SWIR2_mean,SWIR2_std,SWIR2_min,SWIR2_max,SWIR2_median,Red_Edge_mean,Red_Edge_std,Red_Edge_min,Red_Edge_max,Red_Edge_median,Aerosols_mean,Aerosols_std,Aerosols_min,Aerosols_max,Aerosols_median,Red_Edge_2_mean,Red_Edge_2_std,Red_Edge_2_min,Red_Edge_2_max,Red_Edge_2_median,Red_Edge_3_mean,Red_Edge_3_std,Red_Edge_3_min,Red_Edge_3_max,Red_Edge_3_median,Red_Edge_4_mean,Red_Edge_4_std,Red_Edge_4_min,Red_Edge_4_max,Red_Edge_4_median,Water_vapor_mean,Water_vapor_std,Water_vapor_min,Water_vapor_max,Water_vapor_median,Cirrus_mean,Cirrus_std,Cirrus_min,Cirrus_max,Cirrus_median,NDVI_mean,NDVI_std,NDVI_min,NDVI_max,NDVI_median,NDMI_mean,NDMI_std,NDMI_min,NDMI_max,NDMI_median,NDWI_mean,NDWI_std,NDWI_min,NDWI_max,NDWI_median,CI_mean,CI_std,CI_min,CI_max,CI_median
0,1D_0005,0.269574,0.208950,0.0659,0.9038,0.15180,0.289363,0.208852,0.0870,0.9012,0.17310,0.264415,0.232687,0.0404,0.9728,0.13610,0.361094,0.183491,0.0855,0.9232,0.3057,0.251045,0.148545,0.0338,0.6272,0.17990,0.158263,0.115093,0.0149,0.4372,0.10160,0.268254,0.217308,0.0469,0.9236,0.1450,0.311518,0.200160,0.1180,0.8916,0.20250,0.342355,0.195779,0.0788,0.9585,0.2584,0.399212,0.196325,0.1002,1.0377,0.3249,0.426128,0.196539,0.1028,1.0671,0.36630,0.088024,0.075972,0.0150,0.4237,0.06200,0.010221,0.025886,0.0006,0.2416,0.00200,0.278020,0.218983,-0.042087,0.705934,0.296525,0.202949,0.098223,-0.087035,0.610293,0.199651,-0.222745,0.156857,-0.552954,0.007155,-0.209010,0.050226,0.069529,-0.044090,0.270099,0.031455
1,1D_000A,0.271481,0.216468,0.0643,0.9324,0.16015,0.290375,0.217942,0.0895,0.9462,0.17410,0.265776,0.242546,0.0419,0.9789,0.14465,0.398713,0.174216,0.1016,0.9584,0.3558,0.258726,0.146777,0.0394,0.6235,0.19960,0.159674,0.115352,0.0171,0.4858,0.11230,0.271693,0.224892,0.0465,0.9197,0.1561,0.310459,0.205686,0.1159,0.8667,0.20915,0.361773,0.193233,0.0923,0.9400,0.2875,0.427363,0.190140,0.1177,1.0059,0.3760,0.458777,0.189253,0.1218,1.0213,0.41355,0.090193,0.078095,0.0160,0.4680,0.05965,0.009966,0.024263,0.0005,0.2414,0.00215,0.326615,0.261071,-0.031466,0.795084,0.306642,0.240510,0.118951,-0.069135,0.611658,0.263703,-0.275098,0.199968,-0.658319,-0.004489,-0.245730,0.061182,0.083192,-0.051211,0.314374,0.036931
2,1D_000D,0.271598,0.214323,0.0642,0.8980,0.16120,0.292014,0.215681,0.0889,0.9211,0.17960,0.266304,0.238346,0.0428,0.9744,0.14440,0.372353,0.184101,0.0828,0.9425,0.3121,0.253403,0.144844,0.0367,0.6048,0.19110,0.158648,0.111772,0.0177,0.4319,0.10120,0.269136,0.222285,0.0456,0.9430,0.1565,0.312981,0.205051,0.1185,0.8981,0.20760,0.346291,0.198612,0.0739,0.9989,0.2595,0.405110,0.198610,0.0967,1.0820,0.3392,0.433285,0.198448,0.0988,1.1040,0.37760,0.089354,0.077113,0.0157,0.4312,0.05960,0.010001,0.024669,0.0006,0.2378,0.00210,0.292334,0.230403,-0.028932,0.734996,0.301397,0.212733,0.098380,-0.060208,0.612928,0.218087,-0.238293,0.170185,-0.600428,-0.005743,-0.224144,0.046739,0.066896,-0.040166,0.233684,0.028075
3,1D_000E,0.276286,0.215715,0.0665,0.9605,0.16715,0.293892,0.217207,0.0912,0.9601,0.18075,0.269940,0.240166,0.0432,0.9718,0.14510,0.390459,0.180545,0.0909,1.0012,0.3560,0.254516,0.146363,0.0358,0.6323,0.19645,0.157710,0.111949,0.0161,0.4301,0.10935,0.273957,0.222796,0.0457,0.9212,0.1589,0.311448,0.205434,0.1181,0.8574,0.20880,0.354669,0.197582,0.0792,0.9387,0.2838,0.413292,0.198543,0.1010,0.9830,0.3676,0.441320,0.198979,0.0991,1.0051,0.40790,0.089541,0.076872,0.0159,0.4484,0.06225,0.010018,0.024490,0.0006,0.2376,0.00200,0.303352,0.240403,-0.032413,0.759106,0.312223,0.238966,0.110602,-0.011673,0.612466,0.245097,-0.249908,0.180061,-0.613776,0.001104,-0.235535,0.050191,0.071297,-0.039916,0.270006,0.032235
4,1D_0015,0.277222,0.214505,0.0686,0.8280,0.16000,0.294147,0.216074,0.0884,0.8447,0.17520,0.271470,0.242239,0.0429,0.8985,0.14060,0.407985,0.17

In [ ]:
# Number of new generated features
aggregated_train.shape, aggregated_test.shape

((39571, 86), (16960, 86))

In [ ]:
# Add back the train targets to the aggregated df
aggregated_train = aggregated_train.merge(train_targets_df, how = 'left', on = 'ID')

# Preview
aggregated_train.head()

,ID,Green_mean,Green_std,Green_min,Green_max,Green_median,Blue_mean,Blue_std,Blue_min,Blue_max,Blue_median,RED_mean,RED_std,RED_min,RED_max,RED_median,NIR_mean,NIR_std,NIR_min,NIR_max,NIR_median,SWIR1_mean,SWIR1_std,SWIR1_min,SWIR1_max,SWIR1_median,SWIR2_mean,SWIR2_std,SWIR2_min,SWIR2_max,SWIR2_median,Red_Edge_mean,Red_Edge_std,Red_Edge_min,Red_Edge_max,Red_Edge_median,Aerosols_mean,Aerosols_std,Aerosols_min,Aerosols_max,Aerosols_median,Red_Edge_2_mean,Red_Edge_2_std,Red_Edge_2_min,Red_Edge_2_max,Red_Edge_2_median,Red_Edge_3_mean,Red_Edge_3_std,Red_Edge_3_min,Red_Edge_3_max,Red_Edge_3_median,Red_Edge_4_mean,Red_Edge_4_std,Red_Edge_4_min,Red_Edge_4_max,Red_Edge_4_median,Water_vapor_mean,Water_vapor_std,Water_vapor_min,Water_vapor_max,Water_vapor_median,Cirrus_mean,Cirrus_std,Cirrus_min,Cirrus_max,Cirrus_median,NDVI_mean,NDVI_std,NDVI_min,NDVI_max,NDVI_median,NDMI_mean,NDMI_std,NDMI_min,NDMI_max,NDMI_median,NDWI_mean,NDWI_std,NDWI_min,NDWI_max,NDWI_median,CI_mean,CI_std,CI_min,CI_max,CI_median,Target
0,1D_0000,0.265979,0.216543,0.0640,0.8526,0.13740,0.286144,0.217593,0.0859,0.8619,0.15790,0.259690,0.241008,0.0404,0.9137,0.11680,0.367115,0.188226,0.0832,0.9125,0.29625,0.243527,0.149443,0.0322,0.6009,0.16805,0.151610,0.113906,0.0142,0.4322,0.09440,0.263200,0.224953,0.0432,0.8770,0.13665,0.307138,0.205845,0.1181,0.8725,0.19780,0.340618,0.202055,0.0756,0.9129,0.24905,0.399149,0.202998,0.0975,0.9909,0.33135,0.426868,0.202845,0.0933,1.0262,0.36650,0.087540,0.077888,0.0149,0.4551,0.05885,0.009998,0.024321,0.0005,0.2330,0.00190,0.302709,0.232620,-0.031731,0.712057,0.318187,0.231466,0.105327,-0.054230,0.614286,0.254172,-0.243403,0.168868,-0.562095,0.007948,-0.245152,0.051841,0.072480,-0.056774,0.284581,0.037170,0
1,1D_0001,0.270399,0.208321,0.0638,0.8756,0.16675,0.289762,0.209404,0.0845,0.8789,0.18900,0.264174,0.232313,0.0396,0.9464,0.14545,0.379766,0.173545,0.0873,0.8901,0.32950,0.251989,0.142228,0.0332,0.6149,0.19385,0.156691,0.110389,0.0141,0.4389,0.11735,0.267897,0.215786,0.0441,0.8825,0.16175,0.309484,0.198924,0.1168,0.8528,0.20855,0.348151,0.190508,0.0737,0.9015,0.27855,0.408359,0.189984,0.0966,0.9757,0.36140,0.437205,0.190445,0.0992,1.0090,0.39655,0.087958,0.076198,0.0143,0.4441,0.06040,0.010103,0.025336,0.0004,0.2357,0.00205,0.301717,0.238602,-0.050286,0.761315,0.304885,0.229815,0.110257,0.010700,0.611240,0.251263,-0.248447,0.179584,-0.617116,0.026412,-0.227254,0.048843,0.071093,-0.045155,0.236865,0.027336,0
2,1D_0002,0.266084,0.216040,0.0627,0.8610,0.14775,0.286621,0.216439,0.0868,0.8711,0.16585,0.260865,0.241258,0.0398,0.9292,0.12585,0.366290,0.186754,0.0824,0.9086,0.30035,0.244240,0.148668,0.0324,0.5857,0.17620,0.153181,0.113040,0.0146,0.4329,0.10300,0.264412,0.225976,0.0429,0.8909,0.13765,0.307138,0.205845,0.1181,0.8725,0.19780,0.340472,0.203677,0.0718,0.9280,0.25095,0.398143,0.203926,0.0884,1.0060,0.32395,0.426066,0.204229,0.0902,1.0408,0.36305,0.087540,0.077888,0.0149,0.4551,0.05885,0.009998,0.024321,0.0005,0.2330,0.00190,0.299347,0.231626,-0.026485,0.721873,0.309474,0.229222,0.106075,-0.037047,0.612524,0.253078,-0.242578,0.168902,-0.582704,-0.001007,-0.226221,0.049452,0.071069,-0.080468,0.300748,0.034821,0
3,1D_0003,0.266266,0.216637,0.0642,0.8625,0.14630,0.286324,0.216656,0.0860,0.8722,0.16470,0.261431,0.241553,0.0406,0.9350,0.11895,0.361451,0.190320,0.0751,0.9100,0.29475,0.244240,0.148668,0.0324,0.5857,0.17620,0.153181,0.113040,0.0146,0.4329,0.10300,0.264412,0.225976,0.0429,0.8909,0.13765,0.307138,0.205845,0.1181,0.8725,0.19780,0.340472,0.203677,0.0718,0.9280,0.25095,0.398143,0.203926,0.0884,1.0060,0.32395,0.426066,0.204229,0.0902,1.0408,0.36305,0.087540,0.077888,0.0149,0.4551,0.05885,0.009998,0.024321,0.0005,0.2330,0.00190,0.289909,0.225776,-0.047141,0.716433,0.292897,0.219154,0.099812,-0.041223,0.612870,0.237944,-0.233115,0.162751,-0.549183,-0.002660,-0.211015,0.048114,0.066958,-0.040092,0.247040,0.027005,0
4,1D_0004,0.267871,0.215886,0.0632,0.8804,0.16230,0.287685,0.216091,0.0862,0.9002,0.17935,0.262368,0.241414,0.041

In [ ]:
# Split data for training and validation
X_train, X_val, y_train, y_val= train_test_split(aggregated_train, aggregated_train.Target, test_size = 0.2, random_state = 0, stratify = aggregated_train.Target)

# Check shapes of split data
X_train.shape, X_val.shape

((31656, 87), (7915, 87))

In [ ]:
# Check target disribution of split data
X_train.Target.value_counts(normalize = True), X_val.Target.value_counts(normalize = True)

(Target
 0    0.433409
 1    0.335102
 2    0.231489
 Name: proportion, dtype: float64,
 Target
 0    0.433354
 1    0.335186
 2    0.231459
 Name: proportion, dtype: float64)

In [ ]:
X_train.head()

,ID,Green_mean,Green_std,Green_min,Green_max,Green_median,Blue_mean,Blue_std,Blue_min,Blue_max,Blue_median,RED_mean,RED_std,RED_min,RED_max,RED_median,NIR_mean,NIR_std,NIR_min,NIR_max,NIR_median,SWIR1_mean,SWIR1_std,SWIR1_min,SWIR1_max,SWIR1_median,SWIR2_mean,SWIR2_std,SWIR2_min,SWIR2_max,SWIR2_median,Red_Edge_mean,Red_Edge_std,Red_Edge_min,Red_Edge_max,Red_Edge_median,Aerosols_mean,Aerosols_std,Aerosols_min,Aerosols_max,Aerosols_median,Red_Edge_2_mean,Red_Edge_2_std,Red_Edge_2_min,Red_Edge_2_max,Red_Edge_2_median,Red_Edge_3_mean,Red_Edge_3_std,Red_Edge_3_min,Red_Edge_3_max,Red_Edge_3_median,Red_Edge_4_mean,Red_Edge_4_std,Red_Edge_4_min,Red_Edge_4_max,Red_Edge_4_median,Water_vapor_mean,Water_vapor_std,Water_vapor_min,Water_vapor_max,Water_vapor_median,Cirrus_mean,Cirrus_std,Cirrus_min,Cirrus_max,Cirrus_median,NDVI_mean,NDVI_std,NDVI_min,NDVI_max,NDVI_median,NDMI_mean,NDMI_std,NDMI_min,NDMI_max,NDMI_median,NDWI_mean,NDWI_std,NDWI_min,NDWI_max,NDWI_median,CI_mean,CI_std,CI_min,CI_max,CI_median,Target
14283,1D_4FD9,0.286444,0.222804,0.0626,1.0173,0.17370,0.304161,0.226241,0.0817,1.0481,0.18800,0.282394,0.244959,0.0381,1.0553,0.15975,0.398501,0.190808,0.1246,1.0519,0.33560,0.264853,0.136428,0.0500,0.6582,0.2148,0.168929,0.109637,0.0183,0.4318,0.13875,0.286922,0.231424,0.0460,1.0704,0.17465,0.325626,0.216194,0.1105,1.0680,0.22010,0.372904,0.205239,0.0991,1.0715,0.29335,0.432309,0.202128,0.1344,1.0712,0.36495,0.456845,0.200657,0.1420,1.0508,0.39725,0.109647,0.129075,0.0133,0.9930,0.06065,0.023133,0.066272,0.0006,0.5407,0.00245,0.285425,0.225886,-0.025220,0.754217,0.275339,0.213012,0.115823,-0.029135,0.651723,0.206292,-0.241160,0.172447,-0.623389,0.000132,-0.213580,0.043908,0.064904,-0.042543,0.243822,0.026631,2
20993,id_162394a73,0.254394,0.192885,0.0699,0.9512,0.15135,0.268371,0.198763,0.0896,0.9904,0.16455,0.244689,0.214633,0.0448,0.9846,0.12940,0.393708,0.153491,0.1404,0.9967,0.34060,0.286588,0.117731,0.0840,0.6918,0.2367,0.180076,0.102261,0.0372,0.4923,0.12760,0.261116,0.197096,0.0736,1.0050,0.15380,0.291934,0.190864,0.1218,1.0182,0.19565,0.363523,0.168020,0.1258,1.0018,0.29585,0.422604,0.166783,0.1544,1.0139,0.36665,0.456416,0.162721,0.1677,1.0001,0.40600,0.095557,0.103040,0.0229,0.8727,0.06390,0.018548,0.051594,0.0008,0.4205,0.00305,0.336405,0.222719,-0.019211,0.709980,0.358971,0.160259,0.110101,-0.044812,0.673034,0.159456,-0.284483,0.166906,-0.574869,-0.005695,-0.292554,0.083481,0.086303,-0.131981,0.276202,0.071235,1
2588,1D_0E79,0.268118,0.208278,0.0599,0.9236,0.17585,0.289428,0.210785,0.0889,0.9568,0.19585,0.261178,0.230080,0.0348,0.9845,0.16170,0.358971,0.182613,0.0787,0.9925,0.28935,0.238524,0.134127,0.0315,0.6126,0.1900,0.152311,0.109103,0.0122,0.4749,0.10595,0.265578,0.218172,0.0387,0.9486,0.17330,0.315141,0.203968,0.1237,0.9711,0.22030,0.337816,0.197564,0.0748,0.9705,0.25205,0.385567,0.196830,0.0902,1.0583,0.30780,0.410149,0.194719,0.0936,1.0868,0.34210,0.094469,0.113966,0.0103,0.7976,0.04910,0.018639,0.054028,0.0008,0.4405,0.00240,0.265877,0.207510,-0.040585,0.670735,0.260903,0.212636,0.135000,-0.081872,0.742623,0.218859,-0.211873,0.148730,-0.524088,0.010507,-0.193956,0.040903,0.062923,-0.071009,0.226837,0.023277,0
37739,id_7ce74af85,0.266953,0.220258,0.0714,0.9681,0.16690,0.283031,0.225765,0.0846,1.0084,0.17740,0.257977,0.243175,0.0427,1.0058,0.14530,0.390826,0.184123,0.1353,1.0223,0.32330,0.244250,0.123677,0.0348,0.6137,0.1969,0.155237,0.103426,0.0253,0.4719,0.11040,0.268391,0.229608,0.0594,1.0185,0.16280,0.303299,0.219311,0.1101,1.0203,0.19690,0.360483,0.198813,0.1151,1.0198,0.28050,0.418903,0.192147,0.1536,1.0217,0.34770,0.447805,0.187864,0.1644,1.0148,0.38960,0.112513,0.164854,0.0139,0.9338,0.05040,0.031479,0.093561,0.0005,0.5756,0.00210,0.323663,0.238800,-0.046369,0.753891,0.320044,0.236547,0.145221,-0.089383,0.694737,0.223432,-0.267132,0.178343,-0.597795,0.017281,-0.241955,0.069641,0.088661,-0.046214,0.309766,0.042642,1
27156,id_3c1c6a9bf,0.267302,0.205009,0.0834,0.9906,0.15750,0.282467,0.210893,0.0861,1.0773,0

In [ ]:
# Instantiate and train model
model = LGBMClassifier(random_state = 0)

# Prepare data
X_train_data = X_train.drop(['ID', 'Target'], axis = 1)
X_val_data = X_val.drop(['ID', 'Target'], axis = 1)

# Train model
model.fit(X_train_data, y_train)

# Evaluate model
y_preds = model.predict(X_val_data)

# Measure model performance
val_f1_score = f1_score(y_val, y_preds, average = 'macro')
val_f1_score

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035519 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21214
[LightGBM] [Info] Number of data points in the train set: 31656, number of used features: 85
[LightGBM] [Info] Start training from score -0.836073
[LightGBM] [Info] Start training from score -1.093319
[LightGBM] [Info] Start training from score -1.463225


1.0

In [ ]:
# Prepare test data
X_test = aggregated_test.drop(['ID'], axis = 1)

# Make predictions on test
test_preds = model.predict(X_test)

# Prepares sub file
sub = pd.DataFrame()
sub['ID'] = aggregated_test.ID
sub['Target'] = test_preds
sub.head()

,ID,Target
0,1D_0005,0
1,1D_000A,0
2,1D_000D,0
3,1D_000E,0
4,1D_0015,0


In [ ]:
# Check sub target distribution
sub.Target.value_counts(normalize = True)

,proportion
Target,
0,0.433255
1,0.335259
2,0.231486


In [ ]:
# Create submission to be uploaded to Zindi for scoring
sub.to_csv('multiclass_benchmark.csv', index = False)